# Vigloo 숏폼 영상 데이터 → MySQL 저장
`youtube_filtered_332.csv`에서 영상 단위 컬럼만 추출해 `vigloo_shortform` 테이블에 저장

## 0. 환경 설정

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pymysql

load_dotenv()

# MySQL 접속 정보
mysql_host = os.getenv('MYSQL_HOST', 'localhost')
mysql_user = os.getenv('MYSQL_USER', 'root')
mysql_pw   = os.getenv('MYSQL_PASSWORD', '')
mysql_db   = os.getenv('MYSQL_DB', 'youtube_data')

# CSV 경로 (파일 위치에 맞게 수정)
csv_path   = os.path.expanduser('~/Downloads/youtube_filtered_332.csv')
table_name = 'vigloo_shortform'

print(f'MySQL  : {mysql_user}@{mysql_host}/{mysql_db}')
print(f'CSV    : {csv_path}')
print(f'Table  : {table_name}')

MySQL  : root@localhost/youtube_data
CSV    : /Users/jeong-eunseo/Downloads/youtube_filtered_332.csv
Table  : vigloo_shortform


## 1. MySQL 연결

In [2]:
def init_db_engine(host: str, user: str, pw: str, db: str):
    """MySQL DB가 없으면 생성하고 SQLAlchemy 엔진을 반환한다."""
    try:
        # DB 없으면 먼저 생성
        conn = pymysql.connect(host=host, user=user, password=pw, charset='utf8mb4')
        with conn.cursor() as cur:
            cur.execute(
                f'CREATE DATABASE IF NOT EXISTS `{db}` '
                f'CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci'
            )
        conn.close()

        url    = f'mysql+pymysql://{user}:{pw}@{host}/{db}?charset=utf8mb4'
        engine = create_engine(url)

        with engine.connect() as c:
            c.execute(text('SELECT 1'))

        print(f'MySQL 연결 성공: {user}@{host}/{db}')
        return engine
    except Exception as e:
        print(f'[ERROR] MySQL 연결 실패: {e}')
        raise


engine = init_db_engine(mysql_host, mysql_user, mysql_pw, mysql_db)

MySQL 연결 성공: root@localhost/youtube_data


## 2. CSV 읽기

In [3]:
def load_csv(path: str) -> pd.DataFrame:
    """CSV 파일을 읽어 원본 DataFrame을 반환한다. BOM 인코딩(utf-8-sig) 처리 포함."""
    try:
        raw_df = pd.read_csv(path, encoding='utf-8-sig')
        print(f'CSV 로드 완료: {raw_df.shape[0]}행 × {raw_df.shape[1]}열')
        print(f'컬럼 목록: {raw_df.columns.tolist()}')
        return raw_df
    except FileNotFoundError:
        print(f'[ERROR] 파일을 찾을 수 없습니다: {path}')
        raise
    except Exception as e:
        print(f'[ERROR] CSV 로드 실패: {e}')
        raise


raw_df = load_csv(csv_path)
raw_df.head(3)

CSV 로드 완료: 332행 × 16열
컬럼 목록: ['작품명', '검색키워드', 'video_id', '영상제목', '채널명', '업로드일', '조회수', '좋아요수', '댓글수', '영상URL', '댓글작성자', '댓글내용', '댓글좋아요수', '댓글작성일', '긍정', '부정']


,작품명,검색키워드,video_id,영상제목,채널명,업로드일,조회수,좋아요수,댓글수,영상URL,댓글작성자,댓글내용,댓글좋아요수,댓글작성일,긍정,부정
0,국가권력급 상속자가 돌아왔다,국가권력급 상속자가 돌아왔다,-YZcSqqOvnY,식물인간이 된 재벌 상속자 [국가권력급 상속자가 돌아왔다] Ep.1-6,숏드라마는 비글루 (Vigloo),2026-02-26T08:57:14Z,28498,152,9,https://www.youtube.com/watch?v=-YZcSqqOvnY,@그렇지만-n3t,난 이런류가 가장 잼있어 !!! 시원 하다 ~,0.0,2026-03-26T02:14:21Z,False,False
1,국가권력급 상속자가 돌아왔다,국가권력급 상속자가 돌아왔다,-YZcSqqOvnY,식물인간이 된 재벌 상속자 [국가권력급 상속자가 돌아왔다] Ep.1-6,숏드라마는 비글루 (Vigloo),2026-02-26T08:57:14Z,28498,152,9,https://www.youtube.com/watch?v=-YZcSqqOvnY,@on_not_on,남주 이름이 뭐임? 아무리 기억하려고 노력해봐도 기억이 안나네….,0.0,2026-03-19T19:17:25Z,False,False
2,국가권력급 상속자가 돌아왔다,국가권력급 상속자가 돌아왔다,-YZcSqqOvnY,식물인간이 된 재벌 상속자 [국가권력급 상속자가 돌아왔다] Ep.1-6,숏드라마는 비글루 (Vigloo),2026-02-26T08:57:14Z,28498,152,9,https://www.youtube.com/watch?v=-YZcSqqOvnY,@서민욱-w2y,연기도 발연기 지린다...,0.0,2026-03-14T15:16:32Z,False,False


## 3. 컬럼 추출 및 중복 제거

In [4]:
# 저장할 컬럼 목록 (영상 단위 정보만)
TARGET_COLS = [
    '작품명', 'video_id', '영상제목', '채널명',
    '업로드일', '조회수', '좋아요수', '댓글수', '영상URL'
]


def extract_video_df(raw_df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """필요한 컬럼만 추출하고 video_id 기준으로 중복을 제거한다.
    (댓글 행이 여러 개여도 영상 정보는 동일하므로 첫 번째 행 유지)
    """
    try:
        # 컬럼 추출
        video_df = raw_df[cols].copy()

        before_count = len(video_df)

        # video_id 기준 중복 제거 (동일 영상이 댓글 수만큼 반복됨)
        video_df = video_df.drop_duplicates(subset='video_id').reset_index(drop=True)

        after_count = len(video_df)
        print(f'중복 제거: {before_count}행 → {after_count}행 (제거 {before_count - after_count}행)')
        return video_df
    except KeyError as e:
        print(f'[ERROR] 컬럼이 존재하지 않습니다: {e}')
        raise
    except Exception as e:
        print(f'[ERROR] 컬럼 추출 실패: {e}')
        raise


video_df = extract_video_df(raw_df, TARGET_COLS)
video_df.head()

중복 제거: 332행 → 58행 (제거 274행)


,작품명,video_id,영상제목,채널명,업로드일,조회수,좋아요수,댓글수,영상URL
0,국가권력급 상속자가 돌아왔다,-YZcSqqOvnY,식물인간이 된 재벌 상속자 [국가권력급 상속자가 돌아왔다] Ep.1-6,숏드라마는 비글루 (Vigloo),2026-02-26T08:57:14Z,28498,152,9,https://www.youtube.com/watch?v=-YZcSqqOvnY
1,국가권력급 상속자가 돌아왔다,Y1gXXF5ToB0,3년 만에 눈을 뜬 재벌 후계자 [국가권력급 상속자가 돌아왔다] Ep.7-11,숏드라마는 비글루 (Vigloo),2026-02-26T08:02:41Z,59699,533,18,https://www.youtube.com/watch?v=Y1gXXF5ToB0
2,국가권력급 상속자가 돌아왔다,zimg8ScLn94,날 뛰는 폐급 빌런! 정신 번쩍들게 하는 상사의 싸대기,숏드라마는 비글루 (Vigloo),2026-03-09T00:17:03Z,786933,11372,81,https://www.youtube.com/watch?v=zimg8ScLn94
3,국가권력급 상속자가 돌아왔다,OYevmEXAjLU,회장 아들에게 나댄 허세남 참교육 싸대기,숏드라마는 비글루 (Vigloo),2026-03-05T07:58:12Z,484656,5791,21,https://www.youtube.com/watch?v=OYevmEXAjLU
4,국가권력급 상속자가 돌아왔다,nim00qdqCTU,내 아들 건들면 다 죽는거야! 국방부장관의 극대노,숏드라마는 비글루 (Vigloo),2026-03-03T08:55:03Z,87602,1090,8,https://www.youtube.com/watch?v=nim00qdqCTU


## 4. 데이터 타입 정리

In [5]:
def clean_video_df(video_df: pd.DataFrame) -> pd.DataFrame:
    """수치형은 fillna(0) + 정수 변환, 문자형은 fillna('') 처리한다."""
    try:
        df = video_df.copy()

        # 수치형: 결측치 0으로 채우고 정수 변환
        num_cols = ['조회수', '좋아요수', '댓글수']
        for col in num_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

        # 문자형: 결측치 빈 문자열로 채우기
        str_cols = ['작품명', 'video_id', '영상제목', '채널명', '업로드일', '영상URL']
        for col in str_cols:
            df[col] = df[col].fillna('')

        print('데이터 타입 정리 완료')
        print(df.dtypes)
        return df
    except Exception as e:
        print(f'[ERROR] 타입 정리 실패: {e}')
        raise


clean_df = clean_video_df(video_df)
clean_df.head()

데이터 타입 정리 완료
작품명         object
video_id    object
영상제목        object
채널명         object
업로드일        object
조회수          int64
좋아요수         int64
댓글수          int64
영상URL       object
dtype: object


,작품명,video_id,영상제목,채널명,업로드일,조회수,좋아요수,댓글수,영상URL
0,국가권력급 상속자가 돌아왔다,-YZcSqqOvnY,식물인간이 된 재벌 상속자 [국가권력급 상속자가 돌아왔다] Ep.1-6,숏드라마는 비글루 (Vigloo),2026-02-26T08:57:14Z,28498,152,9,https://www.youtube.com/watch?v=-YZcSqqOvnY
1,국가권력급 상속자가 돌아왔다,Y1gXXF5ToB0,3년 만에 눈을 뜬 재벌 후계자 [국가권력급 상속자가 돌아왔다] Ep.7-11,숏드라마는 비글루 (Vigloo),2026-02-26T08:02:41Z,59699,533,18,https://www.youtube.com/watch?v=Y1gXXF5ToB0
2,국가권력급 상속자가 돌아왔다,zimg8ScLn94,날 뛰는 폐급 빌런! 정신 번쩍들게 하는 상사의 싸대기,숏드라마는 비글루 (Vigloo),2026-03-09T00:17:03Z,786933,11372,81,https://www.youtube.com/watch?v=zimg8ScLn94
3,국가권력급 상속자가 돌아왔다,OYevmEXAjLU,회장 아들에게 나댄 허세남 참교육 싸대기,숏드라마는 비글루 (Vigloo),2026-03-05T07:58:12Z,484656,5791,21,https://www.youtube.com/watch?v=OYevmEXAjLU
4,국가권력급 상속자가 돌아왔다,nim00qdqCTU,내 아들 건들면 다 죽는거야! 국방부장관의 극대노,숏드라마는 비글루 (Vigloo),2026-03-03T08:55:03Z,87602,1090,8,https://www.youtube.com/watch?v=nim00qdqCTU


## 5. MySQL 저장

In [6]:
def save_to_mysql(df: pd.DataFrame, engine, table: str) -> None:
    """DataFrame을 MySQL 테이블에 저장한다. 기존 테이블은 대체한다."""
    try:
        df.to_sql(
            name      = table,
            con       = engine,
            if_exists = 'replace',
            index     = False,
            chunksize = 500
        )
        with engine.connect() as conn:
            count = conn.execute(text(f'SELECT COUNT(*) FROM `{table}`')).scalar()
        print(f'저장 완료: {count}개 레코드 → {mysql_db}.{table}')
    except Exception as e:
        print(f'[ERROR] MySQL 저장 실패: {e}')
        raise


save_to_mysql(clean_df, engine, table_name)

저장 완료: 58개 레코드 → youtube_data.vigloo_shortform


## 6. 저장 결과 확인

In [7]:
def load_saved_data(engine, table: str) -> pd.DataFrame:
    """저장된 테이블 전체를 조회해 반환한다."""
    try:
        result_df = pd.read_sql(
            text(f'SELECT * FROM `{table}` ORDER BY 조회수 DESC LIMIT 10'),
            con=engine
        )
        return result_df
    except Exception as e:
        print(f'[ERROR] 결과 조회 실패: {e}')
        raise


result_df = load_saved_data(engine, table_name)
print(f'=== {table_name} 조회수 TOP 10 ===')
display(result_df)

=== vigloo_shortform 조회수 TOP 10 ===


,작품명,video_id,영상제목,채널명,업로드일,조회수,좋아요수,댓글수,영상URL
0,오늘 한류스타와 이혼하겠습니다,OMrPQmsO-8A,며느리 협박하는 막장 시엄니 최후,숏드라마는 비글루 (Vigloo),2026-01-23T08:00:22Z,1605454,16131,36,https://www.youtube.com/watch?v=OMrPQmsO-8A
1,비서할래 와이프할래?,ta5gdh3DAIE,입사시킨 인턴이 내 와이프라고?,숏드라마는 비글루 (Vigloo),2026-03-13T03:01:04Z,1082596,9573,49,https://www.youtube.com/watch?v=ta5gdh3DAIE
2,국가권력급 상속자가 돌아왔다,zimg8ScLn94,날 뛰는 폐급 빌런! 정신 번쩍들게 하는 상사의 싸대기,숏드라마는 비글루 (Vigloo),2026-03-09T00:17:03Z,786933,11372,81,https://www.youtube.com/watch?v=zimg8ScLn94
3,국가권력급 상속자가 돌아왔다,OYevmEXAjLU,회장 아들에게 나댄 허세남 참교육 싸대기,숏드라마는 비글루 (Vigloo),2026-03-05T07:58:12Z,484656,5791,21,https://www.youtube.com/watch?v=OYevmEXAjLU
4,국가권력급 상속자가 돌아왔다,HoTtsYbKILY,아들 괴롭힌 인간들 처단하는 재벌 사모,숏드라마는 비글루 (Vigloo),2026-03-11T08:20:09Z,470747,6221,11,https://www.youtube.com/watch?v=HoTtsYbKILY
5,타임리프 로맨스: 은밀한 복수,vOWvgA6-Uxw,"내 약혼남은 너 가져, 네 남편은 내가 가질게 #비글루 #숏드라마",숏드라마는 비글루 (Vigloo),2025-03-17T10:03:47Z,347171,2864,7,https://www.youtube.com/watch?v=vOWvgA6-Uxw
6,천재 아기의 인생 역전,cZmOemXFpNM,냄새 난다고 손님한테 방향제 뿌리는 무개념 직원,숏드라마는 비글루 (Vigloo),2026-05-11T04:57:08Z,295103,2811,34,https://www.youtube.com/watch?v=cZmOemXFpNM
7,비서할래 와이프할래?,v9mqy6bXg8A,인턴 정체도 모르고 나대는 팀장의 결말은? [비서할래 와이프할래] ep.07-10,숏드라마는 비글루 (Vigloo),2026-03-17T11:30:52Z,212018,1886,114,https://www.youtube.com/watch?v=v9mqy6bXg8A
8,비서할래 와이프할래?,GNE7jDV4Rj8,"5분 만에 계약 결혼, 1년 만에 재회? [비서할래 와이프할래?] Ep.1-6",숏드라마는 비글루 (Vigloo),2026-03-12T07:32:49Z,211556,1735,79,https://www.youtube.com/watch?v=GNE7jDV4Rj8
9,장녀 결혼 일기,yvduruKlVdo,자식을 ATM으로 보던 부모의 최후 #장녀결혼일기,숏드라마는 비글루 (Vigloo),2026-01-28T08:01:08Z,140280,1802,5,https://www.youtube.com/watch?v=yvduruKlVdo


In [8]:
def load_work_summary(engine, table: str) -> pd.DataFrame:
    """작품명별 영상 수와 평균 조회수를 요약해 반환한다."""
    try:
        summary_df = pd.read_sql(
            text(f'''
                SELECT
                    작품명,
                    COUNT(*)            AS 영상수,
                    ROUND(AVG(조회수))  AS 평균조회수,
                    ROUND(AVG(좋아요수)) AS 평균좋아요수,
                    ROUND(AVG(댓글수))  AS 평균댓글수
                FROM `{table}`
                GROUP BY 작품명
                ORDER BY 평균조회수 DESC
            '''),
            con=engine
        )
        return summary_df
    except Exception as e:
        print(f'[ERROR] 작품별 요약 조회 실패: {e}')
        raise


summary_df = load_work_summary(engine, table_name)
print('=== 작품명별 요약 ===')
display(summary_df)

=== 작품명별 요약 ===


,작품명,영상수,평균조회수,평균좋아요수,평균댓글수
0,비서할래 와이프할래?,5,331953.0,2866.0,50.0
1,국가권력급 상속자가 돌아왔다,6,319689.0,4193.0,25.0
2,오늘 한류스타와 이혼하겠습니다,7,292581.0,2695.0,10.0
3,타임리프 로맨스: 은밀한 복수,2,235252.0,1642.0,6.0
4,천재 아기의 인생 역전,5,106279.0,772.0,16.0
5,장녀 결혼 일기,8,85879.0,693.0,7.0
6,수상한 남편의 이중생활,10,50712.0,407.0,5.0
7,닿을 수 있는 그녀,8,48425.0,431.0,5.0
8,뚱보 아내는 조폭 마누라,6,40810.0,434.0,6.0
9,모쏠지옥,1,13542.0,34.0,1.0
